# NB150: The Selection Mechanism — Why These Values, Not Others?

**The question**: The cascade produces amplitudes at all 48 coprime crossings. Reality selects specific patterns as physically realized fermions. What is the selection mechanism WITHIN the dynamics?

**The hypothesis**: Wrapping creates stable excitations. Without wrapping, all perturbations decay exponentially — no persistent structure, no particles. With wrapping, some perturbations become trapped in the periodic potential of the covering circle. They can't fully relax because they've wrapped around and sit in local minima. The physically realized fermions are the **wrapping-stabilized modes** — the patterns that the gradient flow cannot eliminate.

**The investigation**:
1. Run the cascade for LONG times (many primorial periods) and observe which structures persist
2. Identify the modes that decay slowest — these are the "stable excitations"
3. Check whether the persistent modes correspond to the known fermion channels
4. Determine whether the wrapping-stabilized modes are UNIQUE or whether there's a continuous family — if unique, the selection is complete; if continuous, additional structure is needed

**The key shift**: We stop asking "which algebraic label corresponds to which fermion" and instead ask "which dynamical patterns are stable under the gradient flow." The fermions should EMERGE from the dynamics as the long-lived excitations, without being assigned from outside.

## S0: The Long-Time Cascade — What Persists?

The cascade ODE is an overdamped gradient flow toward the solenoid leaf (R_k = 0). Without wrapping, every initial condition decays exponentially at rate κ = 1/√210. The decay time is 1/κ ≈ 14.5 time units. By T = 210 (one primorial period), the transient should be reduced by exp(−κ·210) = exp(−√210) ≈ 6×10⁻⁷.

But this is without wrapping. The covering residuals R_k are angles on circles — they're periodic with period 2π. When a branch's R_k exceeds π, it wraps to the other side of the circle. This wrapping prevents complete relaxation: the branch is now on the "wrong side" of the circle and must traverse the entire 2π to return to zero. The gradient flow pushes toward zero, but the wrapping barrier means some branches get stuck in metastable states.

**First test**: Integrate the cascade for T = 5P₄ = 1050 (five primorial periods) and measure the sector RMS at each window. If wrapping creates persistent structure, the sector RMS should NOT decay to zero — it should stabilize at a nonzero value determined by the wrapping-stabilized modes.

In [3]:
# ── S0: Long-time cascade — what persists? ──

import sys, os, time, numpy as np
from pathlib import Path

ROOT = Path.cwd().parent
if str(ROOT / "scripts") not in sys.path:
    sys.path.insert(0, str(ROOT / "scripts"))

from solenoid_algebra import SA, KAPPA, EPSILON, OMEGA, CP_PAIRS
from solenoid_system import SolenoidSystem
from solenoid_jax import warmup as jax_warmup, detect_device

print('S0: THE LONG-TIME CASCADE — WHAT PERSISTS?')
print('='*70)

sys0 = SolenoidSystem()
primes = SA.primes
P4 = SA.P

# Integrate for T = 5*P4 + 1 = 1051 to get 5 full windows
# Evaluate at ALL coprime crossings across all 5 windows
# That's 48 crossings × 5 windows = 240 evaluation points

n_windows = 5
T_max = float(n_windows * P4 + 1)

# Build evaluation times: ci + w*P4 for each coprime ci, each window w
cis_w0 = SA.coprime_indices(P4)
t_eval_all = []
window_labels = []
ci_labels = []
for w in range(n_windows):
    for ci in cis_w0:
        t_eval_all.append(float(ci + w * P4))
        window_labels.append(w)
        ci_labels.append(ci)

t_eval_all = np.array(t_eval_all)
window_labels = np.array(window_labels)
ci_labels = np.array(ci_labels)

print(f'Integration: T = {T_max:.0f} ({n_windows} primorial windows)')
print(f'Evaluation points: {len(t_eval_all)} ({len(cis_w0)} crossings × {n_windows} windows)')

# JAX integration
print(f'\nJAX device: {detect_device()}')
t0 = time.time()
jax_warmup()
print(f'JAX warmup: {time.time()-t0:.1f}s')

all_branches = sys0.all_branches()
t0 = time.time()
res = sys0.integrate_all_branches(all_branches, t_eval_all, T_max, backend='jax')
dt = time.time() - t0
print(f'Integration: {dt:.1f}s for {len(all_branches)} branches, {len(t_eval_all)} eval pts')

# Extract data: shape (210, n_eval, 4)
R_all = np.zeros((len(all_branches), len(t_eval_all), 4))
for i, br in enumerate(all_branches):
    R_all[i] = res[br]

# Now: compute sector RMS at R3 for each crossing at each window
print(f'\nR3 sector RMS at physical crossings across windows:')
a3_w0, a5_w0, a7_w0 = SA.sector_labels(cis_w0)

# For the 4 physical CP pair crossings
phys_crossings = {'QUARK_g1': 11, 'LEPTON_g1': 31, 'LEPTON_g2': 61, 'QUARK_g2': 191}

for name, ci in phys_crossings.items():
    print(f'\n  {name} (ci={ci}):')
    window_rms = []
    for w in range(n_windows):
        # Find the eval index for this (ci, w)
        idx = np.where((ci_labels == ci) & (window_labels == w))[0][0]
        R3_at_ci = R_all[:, idx, 3]
        R3_wrapped = np.mod(R3_at_ci, 2*np.pi)
        R3_wrapped[R3_wrapped > np.pi] -= 2*np.pi
        rms = np.sqrt(np.mean(R3_wrapped**2))
        window_rms.append(rms)
        
    # Show evolution
    for w in range(n_windows):
        ratio = window_rms[w] / window_rms[0] if window_rms[0] > 0 else 0
        print(f'    Window {w} (t={ci+w*P4:5d}): RMS = {window_rms[w]:.6f}, '
              f'ratio to w0 = {ratio:.6f}')
    
    # Does it decay, grow, or stabilize?
    if abs(window_rms[-1] / window_rms[0] - 1) < 0.001:
        status = 'STABLE (< 0.1% change)'
    elif window_rms[-1] < window_rms[0] * 0.99:
        status = f'DECAYING ({(1-window_rms[-1]/window_rms[0])*100:.2f}% per {n_windows-1} windows)'
    else:
        status = f'GROWING ({(window_rms[-1]/window_rms[0]-1)*100:.2f}% per {n_windows-1} windows)'
    print(f'    Status: {status}')

# Now: the CP ratio across windows
print(f'\n\nCP RATIO STABILITY ACROSS WINDOWS:')
for ch_name, (ch_a3, a7g1, a7g2) in CP_PAIRS.items():
    ci_g1_mask_w0 = (a3_w0 == ch_a3) & (a5_w0 == 0) & (a7_w0 == a7g1)
    ci_g2_mask_w0 = (a3_w0 == ch_a3) & (a5_w0 == 0) & (a7_w0 == a7g2)
    ci_g1 = cis_w0[ci_g1_mask_w0][0]
    ci_g2 = cis_w0[ci_g2_mask_w0][0]
    
    cp_per_window = []
    for w in range(n_windows):
        idx_g1 = np.where((ci_labels == ci_g1) & (window_labels == w))[0][0]
        idx_g2 = np.where((ci_labels == ci_g2) & (window_labels == w))[0][0]
        
        R3_g1 = R_all[:, idx_g1, 3]
        R3_g1_w = np.mod(R3_g1, 2*np.pi)
        R3_g1_w[R3_g1_w > np.pi] -= 2*np.pi
        
        R3_g2 = R_all[:, idx_g2, 3]
        R3_g2_w = np.mod(R3_g2, 2*np.pi)
        R3_g2_w[R3_g2_w > np.pi] -= 2*np.pi
        
        cp = np.sqrt(np.mean(R3_g1_w**2) / np.mean(R3_g2_w**2))
        cp_per_window.append(cp)
    
    print(f'\n  {ch_name} (g1=ci{ci_g1}, g2=ci{ci_g2}):')
    for w in range(n_windows):
        ratio = cp_per_window[w] / cp_per_window[0]
        print(f'    Window {w}: CP = {cp_per_window[w]:.6f}, ratio to w0 = {ratio:.8f}')
    
    cp_spread = (max(cp_per_window) - min(cp_per_window)) / cp_per_window[0]
    print(f'    Spread: {cp_spread*1e6:.1f} ppm')

# The KEY test: at window 4 (t ≈ 4P4), the transient has decayed by
# exp(-κ × 4P4) = exp(-4√210) ≈ 3.7e-26. If the sector RMS is still
# nonzero, it's entirely from the STEADY STATE — not the transient.
# The wrapping-stabilized modes should be visible as the remaining structure.

print(f'\n\nTRANSIENT SUPPRESSION:')
print(f'  exp(-κ × 4P₄) = exp(-4√210) = {np.exp(-4*np.sqrt(P4)):.2e}')
print(f'  By window 4, the transient is negligible (< 10⁻²⁵).')
print(f'  Any remaining sector RMS is PURE STEADY STATE.')
print(f'  The steady state IS the wrapping-stabilized structure.')

S0: THE LONG-TIME CASCADE — WHAT PERSISTS?
Integration: T = 1051 (5 primorial windows)
Evaluation points: 240 (48 crossings × 5 windows)

JAX device: CPU (1 device(s))
JAX warmup: 1.2s
  JAX [CPU (1 device(s))]: 210 branches, 240 eval pts, T=1051.0 — 9.38s
Integration: 9.4s for 210 branches, 240 eval pts

R3 sector RMS at physical crossings across windows:

  QUARK_g1 (ci=11):
    Window 0 (t=   11): RMS = 1.846494, ratio to w0 = 1.000000
    Window 1 (t=  221): RMS = 0.279277, ratio to w0 = 0.151247
    Window 2 (t=  431): RMS = 0.279236, ratio to w0 = 0.151225
    Window 3 (t=  641): RMS = 0.279236, ratio to w0 = 0.151225
    Window 4 (t=  851): RMS = 0.279236, ratio to w0 = 0.151225
    Status: DECAYING (84.88% per 4 windows)

  LEPTON_g1 (ci=31):
    Window 0 (t=   31): RMS = 1.973601, ratio to w0 = 1.000000
    Window 1 (t=  241): RMS = 0.266123, ratio to w0 = 0.134841
    Window 2 (t=  451): RMS = 0.266135, ratio to w0 = 0.134847
    Window 3 (t=  661): RMS = 0.266135, ratio to w

## S1: The Window-0 Imprint — Mass Is a Birth Event

S0 reveals a fundamental truth about the mass mechanism:

**The CP ratio = 1.000000 for all windows except window 0.** By window 1 (t > P₄), the transient has decayed and all crossings have the same steady-state RMS. The mass-generating asymmetry exists ONLY during the first primorial period.

This means fermion masses are NOT properties of persistent excitations. They're properties of the **initial relaxation** — the first pass of the gradient flow from covering misalignment toward the vacuum. The wrapping during this first pass creates the CP asymmetry. After that, it's gone.

**The physical picture**: The solenoid starts misaligned (R_k ≠ 0, set by the branch initial conditions j₁,...,j₄). During the first primorial window, the system relaxes toward alignment. The relaxation creates a transient that interacts with the wrapping threshold at specific coprime crossing times. The crossing times inside the wrapping horizon (ci < 36) see many branches wrap → large RMS compression → large η correction. The crossing times outside see no wrapping → RMS = linear theory → small η. The RATIO of these (the CP pair) encodes the mass.

By window 1, the relaxation is complete. The 210 branches have all converged to the same steady state. The CP ratio is exactly 1. No more mass discrimination.

**This changes the meaning of "mass"**: Mass is not resistance to an ongoing force. Mass is the SIGNATURE of the initial covering misalignment — a birth event that imprints the first-window wrapping pattern onto the crossing-time structure. The mass of a fermion is determined the moment the solenoid begins relaxing, by how its specific CRT position (crossing time) interacts with the transient wrapping during the first primorial period.

In the correspondential reading: mass is not an ongoing struggle. It is the FORM of the first reception — the imprint of the initial condition on the relaxation dynamics. Once the relaxation is complete, the form persists as the mass ratio (frozen in the window-0 CP), even though the transient itself is gone. The mass is a MEMORY of the birth event, not a current process.

## S2: Why Window 0? — The Meaning of t = 0

S1 showed mass is a window-0 imprint — a birth event. But what IS the birth event? In the simulation, t = 0 is when the cascade starts with initial conditions R_k(0) = 2πj_{k+1}. What does this correspond to physically?

**The initial conditions are the solenoid structure itself.** The 210 branches are the 210 sheets of the solenoid. Each sheet starts at a different angular position (2πj_{k+1}/p_{k+1} at each covering level). These positions are not "chosen" — they ARE the solenoid. The solenoid IS the configuration where all sheets are evenly spaced around each covering circle.

**The "birth event" is the onset of dynamics.** At t < 0 (if we could go there), the solenoid is in its initial configuration — all sheets at their natural positions, no coupling, no gradient flow. At t = 0, the dynamics turns on: the covering potential begins pulling the sheets toward alignment, the dissipation begins attenuating the misalignment, and the base frequency ω begins driving the system.

**Window 0 is special because it's the FIRST contact between the solenoid structure and the dynamics.** The initial covering misalignment (2πj_{k+1}) is the maximum possible — the sheets are as far apart as they can be (evenly spaced on a circle of 2π/p_{k+1} per sheet). The first-window transient is the system's response to this maximum misalignment. The wrapping during this response creates the mass-encoding CP asymmetry.

In later windows, the sheets have already relaxed. The covering misalignment is small (steady-state level, ≈ 0.28 radians). No wrapping occurs. No mass discrimination. The system has forgotten its initial structure — it has reached the steady state where all crossings look the same.

**The correspondential reading**: t = 0 is the moment of creation — the beginning of the descent from the celestial (perfect nesting, all sheets distinct) through the spiritual (the covering potential engaging, the gradient flow beginning) to the natural (the steady state, where the initial structure is forgotten and only the mass imprint remains). The mass of each fermion is the TRACE of the original celestial structure that survives the descent into ultimates.

**But is window 0 really unique? Or does each primorial period repeat the same physics?** The driving force ε sin(ωt) is periodic with period 1 (since ω = 2π). The coprime crossings repeat with period P₄ = 210. So every window sees the same steady-state forcing. What makes window 0 different is ONLY the transient — the decaying exponential from the initial conditions. Without the transient, every window is identical (CP = 1).

So the selection mechanism is: **the initial conditions (the solenoid's sheet structure) create a transient that decays during the first primorial window. The interaction of this transient with the wrapping threshold at each coprime crossing time creates the window-0 CP asymmetry. This asymmetry IS the fermion mass spectrum.**

The question then becomes: why THESE initial conditions? Why R_k(0) = 2πj_{k+1}? The answer is: because these are the SOLENOID SHEETS. The initial conditions are not imposed — they ARE the mathematical structure of the covering tower. The 210 sheets of the (2,3,5,7)-solenoid, evenly spaced at each covering level, are the only possible initial conditions for a system that IS a solenoid.

## S3: Initial Condition Dependence — Are the Masses Fixed or Contingent?

If the masses come from the window-0 transient, and the transient comes from the initial conditions, then the masses depend on the initial conditions. But the solenoid has UNIQUE initial conditions: R_k(0) = 2πj_{k+1} for each branch. These are not free parameters — they're the definition of the solenoid's sheet structure.

**Test**: What if we perturb the initial conditions? If R_k(0) = 2πj_{k+1} + δ, do the masses change? How sensitive is the window-0 CP ratio to the initial conditions?

If the CP ratio is SENSITIVE to δ: the masses are contingent on the exact solenoid structure. The specific ICs R_k(0) = 2πj_{k+1} ARE the physics.

If the CP ratio is INSENSITIVE to δ: the masses are robust — the dynamics selects the same mass spectrum regardless of small perturbations. This would mean the wrapping mechanism is structurally stable.

The truth is probably: the masses are sensitive to the SPACING (2π/p_{k+1}) but insensitive to small perturbations (δ ≪ 2π). The covering structure determines the mass spectrum; noise doesn't.

In [7]:
# ── S3: Initial condition sensitivity test ──

print('S3: INITIAL CONDITION DEPENDENCE — ARE MASSES FIXED OR CONTINGENT?')
print('='*70)

# The standard ICs are R_k(0) = 2π j_{k+1}.
# Let me test: what happens if we perturb them?

# We can't easily change ICs in the current framework (they're hardcoded
# as the solenoid sheet structure). But we CAN test by computing the
# CP ratio's sensitivity to the transient amplitude analytically.

# From NB149 S5: CP_full = η(ci_g1) × CP_lin
# where CP_lin = RMS_unwrapped(g1) / RMS_unwrapped(g2)
# and η depends on the wrapping compression.

# The unwrapped RMS at crossing ci is:
# RMS²(ci) = (1/210) Σ_branches (R_k_ss + 2π j_{k+1} α)²
# where α = exp(-κ ci).

# The j_{k+1} contribute 2π j_{k+1} α. If we change the IC to
# R_k(0) = 2π j_{k+1} × s (scaling the sheet spacing by factor s),
# then the transient becomes 2π j_{k+1} s × α.

# At s=1: standard solenoid. At s≠1: perturbed spacing.

# Let me compute the CP ratio as a function of s for the quark channel.
# CP(s) = RMS_wrapped(g1, s) / RMS_wrapped(g2, s)

# For g2 (ci=191), α is negligible → CP_g2 ≈ independent of s
# For g1 (ci=11), α = 0.468 → the transient IS the dominant signal

# I can compute this from the stored R_all data by rescaling the transient.

ci_g1_idx = np.where(cis_w0 == 11)[0][0]
ci_g2_idx = np.where(cis_w0 == 191)[0][0]
alpha_g1 = np.exp(-KAPPA * 11)
alpha_g2 = np.exp(-KAPPA * 191)

# Original j4 values for each branch
j4_vals = np.array([br[3] for br in all_branches])

# Original R3 = R3_ss + 2π j4 α
# Perturbed R3(s) = R3_ss + 2π j4 s α (scaling only the IC, not the SS)

# Extract R3_ss from window 0 data
R3_g1_raw = R_all[:, ci_g1_idx, 3]  # window 0 indices match first 48 of t_eval_all
R3_g1_ss = R3_g1_raw - 2*np.pi * j4_vals * alpha_g1

R3_g2_raw = R_all[:, ci_g2_idx, 3]
R3_g2_ss = R3_g2_raw - 2*np.pi * j4_vals * alpha_g2

print(f'R3_ss at g1 (ci=11): mean={np.mean(R3_g1_ss):.4f}, std={np.std(R3_g1_ss):.4f}')
print(f'R3_ss at g2 (ci=191): mean={np.mean(R3_g2_ss):.4f}, std={np.std(R3_g2_ss):.4f}')
print(f'alpha_g1 = {alpha_g1:.6f}, alpha_g2 = {alpha_g2:.2e}')

# Scan CP(s) for s from 0 to 2
s_values = np.linspace(0.01, 2.0, 200)
cp_quark = []
cp_lepton = []

# Also get lepton crossings
ci_l_g1_idx = np.where(cis_w0 == 31)[0][0]
ci_l_g2_idx = np.where(cis_w0 == 61)[0][0]
alpha_l_g1 = np.exp(-KAPPA * 31)
alpha_l_g2 = np.exp(-KAPPA * 61)

R3_lg1_raw = R_all[:, ci_l_g1_idx, 3]
R3_lg1_ss = R3_lg1_raw - 2*np.pi * j4_vals * alpha_l_g1
R3_lg2_raw = R_all[:, ci_l_g2_idx, 3]
R3_lg2_ss = R3_lg2_raw - 2*np.pi * j4_vals * alpha_l_g2

for s in s_values:
    # Quark
    R3_g1_s = R3_g1_ss + 2*np.pi * j4_vals * s * alpha_g1
    R3_g1_w = np.mod(R3_g1_s, 2*np.pi)
    R3_g1_w[R3_g1_w > np.pi] -= 2*np.pi
    rms_g1 = np.sqrt(np.mean(R3_g1_w**2))
    
    R3_g2_s = R3_g2_ss + 2*np.pi * j4_vals * s * alpha_g2
    R3_g2_w = np.mod(R3_g2_s, 2*np.pi)
    R3_g2_w[R3_g2_w > np.pi] -= 2*np.pi
    rms_g2 = np.sqrt(np.mean(R3_g2_w**2))
    
    cp_q = rms_g1 / rms_g2 if rms_g2 > 1e-10 else 0
    cp_quark.append(cp_q)
    
    # Lepton
    R3_lg1_s = R3_lg1_ss + 2*np.pi * j4_vals * s * alpha_l_g1
    R3_lg1_w = np.mod(R3_lg1_s, 2*np.pi)
    R3_lg1_w[R3_lg1_w > np.pi] -= 2*np.pi
    rms_lg1 = np.sqrt(np.mean(R3_lg1_w**2))
    
    R3_lg2_s = R3_lg2_ss + 2*np.pi * j4_vals * s * alpha_l_g2
    R3_lg2_w = np.mod(R3_lg2_s, 2*np.pi)
    R3_lg2_w[R3_lg2_w > np.pi] -= 2*np.pi
    rms_lg2 = np.sqrt(np.mean(R3_lg2_w**2))
    
    cp_l = rms_lg1 / rms_lg2 if rms_lg2 > 1e-10 else 0
    cp_lepton.append(cp_l)

cp_quark = np.array(cp_quark)
cp_lepton = np.array(cp_lepton)

# Find the CP at s=1 (standard)
s1_idx = np.argmin(np.abs(s_values - 1.0))
print(f'\nCP(s=1): quark = {cp_quark[s1_idx]:.4f}, lepton = {cp_lepton[s1_idx]:.4f}')

# How does CP vary with s?
print(f'\nCP sensitivity to IC scaling s:')
print(f'  {"s":>6s}  {"CP_quark":>10s}  {"CP_lepton":>10s}')
for s_show in [0.1, 0.5, 0.8, 0.9, 1.0, 1.1, 1.2, 1.5, 2.0]:
    idx = np.argmin(np.abs(s_values - s_show))
    print(f'  {s_values[idx]:6.2f}  {cp_quark[idx]:10.4f}  {cp_lepton[idx]:10.4f}')

# The derivative dCP/ds at s=1
ds = s_values[1] - s_values[0]
dcp_q = np.gradient(cp_quark, ds)
dcp_l = np.gradient(cp_lepton, ds)
sens_q = dcp_q[s1_idx] / cp_quark[s1_idx]  # relative sensitivity
sens_l = dcp_l[s1_idx] / cp_lepton[s1_idx]

print(f'\n  Relative sensitivity at s=1:')
print(f'    Quark: (1/CP)(dCP/ds) = {sens_q:.4f} (a 1% change in s gives {abs(sens_q):.2f}% change in CP)')
print(f'    Lepton: (1/CP)(dCP/ds) = {sens_l:.4f} (a 1% change in s gives {abs(sens_l):.2f}% change in CP)')

# Is there anything special about s=1?
# Is CP(s) monotonic? Does it have a maximum/minimum?
s_max_q = s_values[np.argmax(cp_quark)]
s_max_l = s_values[np.argmax(cp_lepton)]
print(f'\n  CP maximum locations:')
print(f'    Quark: max CP at s = {s_max_q:.2f} (CP = {np.max(cp_quark):.4f})')
print(f'    Lepton: max CP at s = {s_max_l:.2f} (CP = {np.max(cp_lepton):.4f})')

# As s → 0: no transient → CP → 1 (both g1 and g2 at steady state)
# As s → ∞: all branches wrapped → CP → (wrapped RMS) / (steady state)
print(f'\n  Limits:')
print(f'    s → 0: CP_q → {cp_quark[0]:.4f}, CP_l → {cp_lepton[0]:.4f} (should → RMS_ss/RMS_ss ≈ 1)')
print(f'    s → 2: CP_q → {cp_quark[-1]:.4f}, CP_l → {cp_lepton[-1]:.4f}')

S3: INITIAL CONDITION DEPENDENCE — ARE MASSES FIXED OR CONTINGENT?
R3_ss at g1 (ci=11): mean=0.8713, std=0.3186
R3_ss at g2 (ci=191): mean=0.2795, std=0.0001
alpha_g1 = 0.468101, alpha_g2 = 1.89e-06

CP(s=1): quark = 6.6067, lepton = 5.9120

CP sensitivity to IC scaling s:
       s    CP_quark   CP_lepton
    0.10      6.6957      7.4117
    0.50      6.8884      9.7169
    0.80      6.6313      7.6295
    0.90      6.5403      6.7346
    1.00      6.6067      5.9120
    1.10      5.7596      5.1998
    1.20      6.5169      4.5867
    1.50      6.4255      3.4030
    2.00      3.3061      2.8615

  Relative sensitivity at s=1:
    Quark: (1/CP)(dCP/ds) = -0.1460 (a 1% change in s gives 0.15% change in CP)
    Lepton: (1/CP)(dCP/ds) = -1.2970 (a 1% change in s gives 1.30% change in CP)

  CP maximum locations:
    Quark: max CP at s = 0.16 (CP = 7.8833)
    Lepton: max CP at s = 0.41 (CP = 9.8423)

  Limits:
    s → 0: CP_q → 3.6240, CP_l → 5.9435 (should → RMS_ss/RMS_ss ≈ 1)
    s → 2